# Wind Farm C Federated Dataset Builder

This notebook prepares only Wind Farm C from CARE to Compare.

**Design:** one `asset_id` = one federated-learning client. Events are processed separately, timestamps are sorted within each event, validation is taken chronologically from the source training period, and `event_id` is retained so later windows cannot cross event boundaries.

Raw event copying and ZIP creation are disabled by default to prevent Kaggle disk exhaustion.

In [1]:
from pathlib import Path
import re, json, shutil
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 15)
pd.set_option("display.max_rows", 30)

INPUT_ROOT = Path("/kaggle/input")
OUTPUT_ROOT = Path("/kaggle/working/federated_windfarm_c")
VALIDATION_FRACTION = 0.15
MIN_TRAIN_ROWS_FOR_VALIDATION = 20
COPY_RAW_EVENTS = False
MAKE_ZIP = False
OVERWRITE_OUTPUT = True

print("Configuration loaded.")

Configuration loaded.


### Cell 1 — Imports and configuration

This cell imports the libraries and defines the experiment settings.

The notebook searches only `/kaggle/input/`, selects Farm C files, and writes results to `/kaggle/working/federated_windfarm_c`.

`COPY_RAW_EVENTS=False` and `MAKE_ZIP=False` are intentional: the previous version exhausted Kaggle storage by duplicating large files and creating an archive.

In [2]:
def human_bytes(n):
    units = ["B", "KB", "MB", "GB", "TB"]
    n = float(n)
    for unit in units:
        if n < 1024 or unit == units[-1]:
            return f"{n:.2f} {unit}"
        n /= 1024

def show_disk_space(path="/kaggle/working"):
    u = shutil.disk_usage(path)
    print("Free disk:", human_bytes(u.free))
    return u

show_disk_space()

Free disk: 19.50 GB


usage(total=20957446144, used=151552, free=20940517376)

### Cell 2 — Disk-space check

This cell checks free space before processing. It is a safeguard against the earlier `No space left on device` failure.

The revised workflow processes one event at a time and does not create a ZIP unless explicitly enabled.

In [3]:
def path_text(p):
    return str(p).replace("\\", "/").lower()

def is_farm_c(p):
    t = path_text(p)
    return (
        "/wind farm c/" in t or
        "/wind_farm_c/" in t or
        "/windfarm_c/" in t or
        re.search(r"(^|/)(farm[_ ]?c)(/|$)", t) is not None
    )

all_csvs = sorted(INPUT_ROOT.rglob("*.csv"))
farm_c_csvs = [p for p in all_csvs if is_farm_c(p)]

print("All CSVs:", len(all_csvs))
print("Farm C candidates:", len(farm_c_csvs))
for p in farm_c_csvs[:20]:
    print(p)

All CSVs: 202
Farm C candidates: 120
/kaggle/input/datasets/azizkasimov/wind-turbine-scada-data-for-early-fault-detection/Wind Farm C/comma_event_info.csv
/kaggle/input/datasets/azizkasimov/wind-turbine-scada-data-for-early-fault-detection/Wind Farm C/comma_feature_description.csv
/kaggle/input/datasets/azizkasimov/wind-turbine-scada-data-for-early-fault-detection/Wind Farm C/datasets/1.csv
/kaggle/input/datasets/azizkasimov/wind-turbine-scada-data-for-early-fault-detection/Wind Farm C/datasets/11.csv
/kaggle/input/datasets/azizkasimov/wind-turbine-scada-data-for-early-fault-detection/Wind Farm C/datasets/12.csv
/kaggle/input/datasets/azizkasimov/wind-turbine-scada-data-for-early-fault-detection/Wind Farm C/datasets/15.csv
/kaggle/input/datasets/azizkasimov/wind-turbine-scada-data-for-early-fault-detection/Wind Farm C/datasets/16.csv
/kaggle/input/datasets/azizkasimov/wind-turbine-scada-data-for-early-fault-detection/Wind Farm C/datasets/18.csv
/kaggle/input/datasets/azizkasimov/wind-t

### Cell 3 — Farm C file discovery

This cell scans the Kaggle input directory but keeps only paths belonging to Wind Farm C.

It does not assume one exact Kaggle folder name because dataset mounts can vary. The output is only a candidate list; the next cell checks headers.

In [4]:
def header_of(p):
    try:
        return list(pd.read_csv(p, nrows=0).columns)
    except Exception as e:
        return None

event_files, rejected = [], []

for p in farm_c_csvs:
    h = header_of(p)
    if h is not None and {"time_stamp", "asset_id"}.issubset(set(map(str, h))):
        event_files.append(p)
    else:
        rejected.append(str(p))

print("Accepted event files:", len(event_files))
print("Rejected files:", len(rejected))
print("First rejected files:", rejected[:10])

Accepted event files: 58
Rejected files: 62
First rejected files: ['/kaggle/input/datasets/azizkasimov/wind-turbine-scada-data-for-early-fault-detection/Wind Farm C/comma_event_info.csv', '/kaggle/input/datasets/azizkasimov/wind-turbine-scada-data-for-early-fault-detection/Wind Farm C/comma_feature_description.csv', '/kaggle/input/datasets/azizkasimov/wind-turbine-scada-data-for-early-fault-detection/Wind Farm C/datasets/1.csv', '/kaggle/input/datasets/azizkasimov/wind-turbine-scada-data-for-early-fault-detection/Wind Farm C/datasets/11.csv', '/kaggle/input/datasets/azizkasimov/wind-turbine-scada-data-for-early-fault-detection/Wind Farm C/datasets/12.csv', '/kaggle/input/datasets/azizkasimov/wind-turbine-scada-data-for-early-fault-detection/Wind Farm C/datasets/15.csv', '/kaggle/input/datasets/azizkasimov/wind-turbine-scada-data-for-early-fault-detection/Wind Farm C/datasets/16.csv', '/kaggle/input/datasets/azizkasimov/wind-turbine-scada-data-for-early-fault-detection/Wind Farm C/datas

### Cell 4 — Header validation

Only files containing both `time_stamp` and `asset_id` are accepted as event files.

This prevents metadata CSVs or unrelated files from entering the pipeline. Reading `nrows=0` checks the header without loading the full 957-feature dataset.

In [5]:
if OUTPUT_ROOT.exists() and OVERWRITE_OUTPUT:
    shutil.rmtree(OUTPUT_ROOT)
(OUTPUT_ROOT / "clients").mkdir(parents=True, exist_ok=True)

def safe_token(x):
    x = re.sub(r"[^A-Za-z0-9_.-]+", "_", str(x).strip())
    return x[:100] or "unknown"

def event_id_from_path(p):
    return p.stem

def client_dir(asset_id):
    return OUTPUT_ROOT / "clients" / f"client_asset_{safe_token(asset_id)}"

def append_csv(df, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, mode="a", header=not path.exists(), index=False)

print("Output directory prepared.")

Output directory prepared.


### Cell 5 — Output and helper functions

This cell creates the output directory and defines helpers for safe folder names and appending data.

Appending is necessary because several event files can belong to the same turbine/client. The data is written incrementally rather than collecting all Farm C data in memory.

In [6]:
def normalize_split(s):
    x = s.astype(str).str.strip().str.lower()
    train_values = {"train", "training", "0", "false", "fit"}
    test_values = {"test", "testing", "prediction", "predict", "1", "true"}
    out = pd.Series("unknown", index=s.index)
    out[x.isin(train_values)] = "train"
    out[x.isin(test_values)] = "test"
    return out

def chronological(df):
    t = pd.to_datetime(df["time_stamp"], errors="coerce")
    out = df.copy()
    out["__time"] = t
    out = out.sort_values("__time", kind="mergesort", na_position="last")
    return out.drop(columns="__time")

def split_event(df):
    if "train_test" not in df.columns:
        raise ValueError("train_test column missing")

    df = chronological(df)
    labels = normalize_split(df["train_test"])
    source_train = df[labels == "train"].copy()
    source_test = df[labels == "test"].copy()

    if source_train.empty:
        raise ValueError("No source training rows")

    n_val = 0
    if len(source_train) >= MIN_TRAIN_ROWS_FOR_VALIDATION:
        n_val = max(1, int(np.floor(len(source_train) * VALIDATION_FRACTION)))
        n_val = min(n_val, len(source_train) - 1)

    train = source_train.iloc[:-n_val] if n_val else source_train
    val = source_train.iloc[-n_val:] if n_val else source_train.iloc[0:0]
    return train.copy(), val.copy(), source_test.copy()

### Cell 6 — Leakage-aware split functions

This cell defines the temporal split.

For each event:

1. sort by `time_stamp`;
2. use the original `train_test` field;
3. take the final 15% of source training rows as validation;
4. retain prediction/test rows as test.

Validation is not randomly sampled. This preserves the time direction and better represents future prediction. The split is done separately per event.

In [7]:
manifest_rows, skipped = [], []
schema_columns = []

for p in event_files:
    eid = event_id_from_path(p)
    print("Processing:", p.name)

    try:
        df = pd.read_csv(p)
        if not {"time_stamp", "asset_id"}.issubset(df.columns):
            raise ValueError("Required columns missing")

        for c in df.columns:
            if c not in schema_columns:
                schema_columns.append(c)

        df["event_id"] = eid
        df["farm"] = "C"

        train, val, test = split_event(df)

        for asset in sorted(df["asset_id"].dropna().astype(str).unique()):
            tr = train[train["asset_id"].astype(str) == asset]
            va = val[val["asset_id"].astype(str) == asset]
            te = test[test["asset_id"].astype(str) == asset]

            d = client_dir(asset)
            append_csv(tr, d / "train.csv")
            append_csv(va, d / "validation.csv")
            append_csv(te, d / "test.csv")

            manifest_rows.append({
                "client_id": f"asset_{safe_token(asset)}",
                "asset_id": asset,
                "farm": "C",
                "event_id": eid,
                "source_file": str(p),
                "train_rows": len(tr),
                "validation_rows": len(va),
                "test_rows": len(te)
            })

        if COPY_RAW_EVENTS:
            raw = OUTPUT_ROOT / "raw_events"
            raw.mkdir(exist_ok=True)
            shutil.copy2(p, raw / p.name)

    except Exception as e:
        skipped.append({"path": str(p), "event_id": eid, "error": repr(e)})
        print("  SKIPPED:", repr(e))

manifest = pd.DataFrame(manifest_rows)
if manifest.empty:
    raise RuntimeError("No data was processed.")

manifest.to_csv(OUTPUT_ROOT / "client_manifest.csv", index=False)

print("Clients:", manifest.client_id.nunique())
print("Events:", manifest.event_id.nunique())
print("Skipped:", len(skipped))

Processing: comma_1.csv
Processing: comma_11.csv
Processing: comma_12.csv
Processing: comma_15.csv
Processing: comma_16.csv
Processing: comma_18.csv
Processing: comma_20.csv
Processing: comma_28.csv
Processing: comma_29.csv
Processing: comma_30.csv
Processing: comma_31.csv
Processing: comma_32.csv
Processing: comma_33.csv
Processing: comma_35.csv
Processing: comma_36.csv
Processing: comma_37.csv
Processing: comma_39.csv
Processing: comma_4.csv
Processing: comma_41.csv
Processing: comma_43.csv
Processing: comma_44.csv
Processing: comma_46.csv
Processing: comma_47.csv
Processing: comma_48.csv
Processing: comma_49.csv
Processing: comma_5.csv
Processing: comma_50.csv
Processing: comma_54.csv
Processing: comma_55.csv
Processing: comma_56.csv
Processing: comma_57.csv
Processing: comma_58.csv
Processing: comma_59.csv
Processing: comma_6.csv
Processing: comma_60.csv
Processing: comma_61.csv
Processing: comma_62.csv
Processing: comma_63.csv
Processing: comma_64.csv
Processing: comma_65.csv
Proc

### Cell 7 — Main processing loop

This is the core cell.

Each Farm C event file is loaded one at a time. The code adds `event_id` and `farm`, performs the chronological split, groups by `asset_id`, and appends rows to the correct client files.

Because `event_id` is stored in every row, later sequence generation can prevent windows from crossing event boundaries.

In [8]:
schema = {
    "farm": "Wind Farm C",
    "columns_seen": schema_columns,
    "required_columns": ["time_stamp", "asset_id"],
    "validation_fraction": VALIDATION_FRACTION,
    "client_definition": "One asset_id in Wind Farm C equals one FL client",
    "raw_events_copied": COPY_RAW_EVENTS
}
with open(OUTPUT_ROOT / "feature_schema.json", "w", encoding="utf-8") as f:
    json.dump(schema, f, indent=2)

summary = (
    manifest.groupby(["client_id", "asset_id"], as_index=False)
    .agg(
        events=("event_id", "nunique"),
        train_rows=("train_rows", "sum"),
        validation_rows=("validation_rows", "sum"),
        test_rows=("test_rows", "sum")
    )
    .sort_values("asset_id")
)
summary.to_csv(OUTPUT_ROOT / "client_summary.csv", index=False)

print("Client summary:")
display(summary)

Client summary:


,client_id,asset_id,events,train_rows,validation_rows,test_rows
0,asset_12,12,3,131826,23262,8001
1,asset_13,13,1,44676,7884,2305
2,asset_14,14,1,43208,7624,1585
3,asset_15,15,2,89352,15768,2459
4,asset_16,16,4,178704,31536,10203
5,asset_2,2,2,89352,15768,6435
6,asset_21,21,2,89108,15724,2609
7,asset_23,23,3,132927,23457,8280
8,asset_32,32,2,88618,15638,3404
9,asset_33,33,2,87394,15422,5864


### Cell 8 — Manifest, schema, and client summary

This cell writes three useful artifacts:

- `client_manifest.csv`: one row per client-event combination;
- `feature_schema.json`: columns and split policy;
- `client_summary.csv`: total rows and event counts per turbine.

These files allow you to audit the federated partition without opening every large CSV.

In [9]:
if skipped:
    with open(OUTPUT_ROOT / "skipped_files.json", "w", encoding="utf-8") as f:
        json.dump(skipped, f, indent=2)

readme = f'''# Federated Wind Farm C Dataset

One asset_id from Wind Farm C is one FL client.

Train uses source training rows excluding the final {VALIDATION_FRACTION:.0%}.
Validation uses the final chronological portion of source training rows.
Test uses source prediction/test rows.

event_id is retained to prevent cross-event windows.
Raw events copied: {COPY_RAW_EVENTS}
'''

(OUTPUT_ROOT / "README.md").write_text(readme, encoding="utf-8")
print("Documentation written.")

Documentation written.


### Cell 9 — Diagnostics and documentation

This cell saves any skipped-file errors and writes a README.

A skipped-file report is important: it prevents an event from silently disappearing. The README records the client definition and split policy so the generated dataset remains understandable later.

In [10]:
# Basic integrity checks
assert manifest["farm"].eq("C").all()
assert manifest["client_id"].notna().all()
assert (manifest[["train_rows", "validation_rows", "test_rows"]] >= 0).all().all()

print("Integrity checks passed.")
print("Output:", OUTPUT_ROOT)

for p in sorted(OUTPUT_ROOT.rglob("*")):
    if p.is_file():
        print(" -", p.relative_to(OUTPUT_ROOT))

Integrity checks passed.
Output: /kaggle/working/federated_windfarm_c
 - README.md
 - client_manifest.csv
 - client_summary.csv
 - clients/client_asset_12/test.csv
 - clients/client_asset_12/train.csv
 - clients/client_asset_12/validation.csv
 - clients/client_asset_13/test.csv
 - clients/client_asset_13/train.csv
 - clients/client_asset_13/validation.csv
 - clients/client_asset_14/test.csv
 - clients/client_asset_14/train.csv
 - clients/client_asset_14/validation.csv
 - clients/client_asset_15/test.csv
 - clients/client_asset_15/train.csv
 - clients/client_asset_15/validation.csv
 - clients/client_asset_16/test.csv
 - clients/client_asset_16/train.csv
 - clients/client_asset_16/validation.csv
 - clients/client_asset_2/test.csv
 - clients/client_asset_2/train.csv
 - clients/client_asset_2/validation.csv
 - clients/client_asset_21/test.csv
 - clients/client_asset_21/train.csv
 - clients/client_asset_21/validation.csv
 - clients/client_asset_23/test.csv
 - clients/client_asset_23/train.c

### Cell 10 — Final integrity check

This cell verifies that all manifest rows belong to Farm C, identifiers exist, and row counts are valid.

It then lists the generated files. Run this before using the output in your FL training notebook.

In [11]:
def make_windows(df, feature_columns, window_size=144, stride=1):
    if "event_id" not in df.columns:
        raise ValueError("event_id is required")

    windows = []
    for event_id, event_df in df.groupby("event_id", sort=False):
        event_df = chronological(event_df)
        values = event_df[feature_columns].to_numpy(dtype=np.float32)

        for start in range(0, len(values) - window_size + 1, stride):
            windows.append(values[start:start + window_size])

    return np.asarray(windows, dtype=np.float32)

print("Leakage-safe windowing helper defined.")

Leakage-safe windowing helper defined.


### Cell 11 — Optional windowing helper

This helper is for the next notebook stage.

It groups by `event_id` before creating sliding windows. This is essential: a window must never combine the end of one event with the beginning of another event.

`window_size=144` represents 24 hours at 10-minute resolution, but you can change it for your experiment.

In [12]:
if MAKE_ZIP:
    show_disk_space()
    archive = shutil.make_archive(
        str(OUTPUT_ROOT),
        "zip",
        root_dir=str(OUTPUT_ROOT.parent),
        base_dir=OUTPUT_ROOT.name
    )
    print("ZIP created:", archive)
else:
    print("ZIP creation skipped.")

ZIP creation skipped.


### Cell 12 — Optional ZIP creation

ZIP creation is disabled by default because it can temporarily require substantial extra disk space.

Enable `MAKE_ZIP=True` only after confirming that Kaggle has enough free storage.